# SkinSense — Train EfficientNet-B0 (12 skin conditions)

Self-contained trainer. Consumes the **final merged dataset** in `data/`:

```
data/train/<class>/*.jpg
data/val/<class>/*.jpg
```

The architecture here is **identical** to the backend's `model_loader.py`, so the
saved checkpoint drops straight into inference — set `WEIGHTS_PATH` to it.

**Colab:** Runtime → Change runtime type → **GPU**, upload `data.zip` (see the
"Colab data upload" cell), then run top to bottom.


In [ ]:
# --- deps (Colab already has torch/torchvision; harmless locally) ---
!pip -q install torch torchvision tqdm pillow

In [ ]:
# --- (Colab only) upload & unzip the dataset ---
# Locally, skip this cell — data/ is already beside the notebook.
# On Colab:  zip your local data/ folder first (see last cell), upload data.zip,
# then run:
#   from google.colab import files; files.upload()   # pick data.zip
#   !unzip -q data.zip -d .
import os
print("data/ present:", os.path.isdir("data"))

In [ ]:
import logging, sys
from pathlib import Path
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import models, datasets, transforms

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
log = logging.getLogger("train")

# ---- must match backend/app/ml/model_loader.py CLASS_NAMES (order = output) ----
CLASS_NAMES = [
    "acne", "eczema", "psoriasis", "rosacea", "seborrheic_keratoses", "tinea",
    "melasma", "vitiligo", "hyperpigmentation", "contact_dermatitis", "warts",
    "actinic_keratosis",
]

# ---- config ----
DATA   = Path("./data")
OUT    = "skinsense_efficientnet_b0.pt"
EPOCHS = 25
WARMUP_EPOCHS = 3     # head-only epochs before unfreezing the backbone
BATCH  = 32
LR     = 3e-4
WD     = 1e-4
WORKERS = 2
PATIENCE = 6
LABEL_SMOOTHING = 0.05

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
log.info("Device: %s", device)

In [ ]:
# ---- architecture (mirrors model_loader.py so checkpoints load strict=True) ----
class TemperatureScaler(nn.Module):
    def __init__(self, base_model, temperature=1.0):
        super().__init__()
        self.base_model = base_model
        self.temperature = nn.Parameter(torch.tensor(float(temperature)), requires_grad=False)
    def forward(self, x):
        return self.base_model(x) / self.temperature

def build_efficientnet_b0(num_classes=len(CLASS_NAMES), pretrained=True):
    weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None
    net = models.efficientnet_b0(weights=weights)
    net.classifier[1] = nn.Linear(net.classifier[1].in_features, num_classes)
    return net

In [ ]:
# ---- data ----
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20), transforms.ColorJitter(0.2, 0.2, 0.2, 0.02),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD),
])
val_tf = transforms.Compose([  # mirrors inference preprocessing
    transforms.Resize((224, 224)), transforms.ToTensor(), transforms.Normalize(MEAN, STD),
])

train_ds = datasets.ImageFolder(DATA / "train", transform=train_tf)
val_ds   = datasets.ImageFolder(DATA / "val",   transform=val_tf)

# Remap ImageFolder's local (alphabetical, possibly-fewer) labels into the global
# 12-class CLASS_NAMES index, so a subset-trained checkpoint still loads at serve.
unknown = set(train_ds.classes) - set(CLASS_NAMES)
assert not unknown, f"folders not in CLASS_NAMES: {sorted(unknown)}"
for ds in (train_ds, val_ds):
    remap = {li: CLASS_NAMES.index(n) for n, li in ds.class_to_idx.items()}
    ds.target_transform = lambda y, _m=remap: _m[y]

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=WORKERS, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=WORKERS, pin_memory=True)
log.info("present folders (%d): %s", len(train_ds.classes), train_ds.classes)
log.info("train=%d  val=%d", len(train_ds), len(val_ds))

In [ ]:
# ---- inverse-frequency class weights over the global 12 slots ----
# Absent classes get weight 0 (otherwise huge inverse-freq weights + label
# smoothing collapse predictions onto classes that never appear).
local_to_global = {li: CLASS_NAMES.index(n) for n, li in train_ds.class_to_idx.items()}
counts = torch.zeros(len(CLASS_NAMES))
for _, ll in train_ds.samples:
    counts[local_to_global[ll]] += 1
present = counts > 0
w = torch.zeros(len(CLASS_NAMES))
w[present] = counts[present].sum() / (present.sum() * counts[present])
class_weights = w.to(device)
for c, n in zip(CLASS_NAMES, counts.tolist()):
    log.info("  %-22s %d", c, int(n))

In [ ]:
# ---- eval + calibration helpers ----
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    n = len(CLASS_NAMES); correct = total = 0
    pc = torch.zeros(n); pt = torch.zeros(n)
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        pred = model(x).argmax(1)
        correct += (pred == y).sum().item(); total += y.numel()
        for c in range(n):
            m = y == c; pt[c] += m.sum().item(); pc[c] += (pred[m] == c).sum().item()
    acc = correct / max(total, 1)
    recalls = pc / pt.clamp(min=1)
    macro = recalls[pt > 0].mean().item()
    return acc, macro

def calibrate_temperature(model, loader):
    model.eval(); L, Y = [], []
    with torch.no_grad():
        for x, y in loader:
            L.append(model.base_model(x.to(device)).cpu()); Y.append(y)
    if not L: return 1.0
    logits, labels = torch.cat(L), torch.cat(Y)
    T = torch.nn.Parameter(torch.ones(1)); opt = torch.optim.LBFGS([T], lr=0.01, max_iter=100)
    def closure():
        opt.zero_grad(); loss = F.cross_entropy(logits / T.clamp(min=0.05), labels)
        loss.backward(); return loss
    opt.step(closure)
    raw = float(T.detach().item()); t = min(max(raw, 0.5), 5.0)
    if t in (0.5, 5.0):
        log.warning("calibration hit bound (raw=%.3f); using T=1.0", raw); t = 1.0
    log.info("calibrated temperature = %.3f (raw %.3f)", t, raw)
    return t

def set_backbone_trainable(model, trainable):
    for name, p in model.base_model.named_parameters():
        p.requires_grad = True if name.startswith("classifier") else trainable

In [ ]:
# ---- train ----
model = TemperatureScaler(build_efficientnet_b0(pretrained=True), temperature=1.0).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=LABEL_SMOOTHING)
optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR, weight_decay=WD)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
use_amp = device.type == "cuda"
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

set_backbone_trainable(model, False)  # warmup: head only
best, no_improve = -1.0, 0
for epoch in range(1, EPOCHS + 1):
    if epoch == WARMUP_EPOCHS + 1:
        log.info("unfreezing backbone"); set_backbone_trainable(model, True)
        optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, EPOCHS - epoch + 1))
    model.train(); running = 0.0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=use_amp):
            loss = criterion(model(x), y)
        scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
        running += loss.item() * y.size(0)
    scheduler.step()
    acc, macro = evaluate(model, val_loader)
    log.info("epoch %2d/%d  loss=%.4f  val_acc=%.4f  val_macro_recall=%.4f",
             epoch, EPOCHS, running / max(len(train_ds), 1), acc, macro)
    if macro > best:
        best, no_improve = macro, 0
        torch.save(model.state_dict(), OUT); log.info("  saved best -> %s (macro=%.4f)", OUT, macro)
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            log.info("early stopping"); break

In [ ]:
# ---- reload best, calibrate temperature on val, re-save ----
model.load_state_dict(torch.load(OUT, map_location=device))
t = calibrate_temperature(model, val_loader)
with torch.no_grad():
    model.temperature.copy_(torch.tensor(float(t)))
torch.save(model.state_dict(), OUT)
log.info("done. best macro_recall=%.4f  checkpoint=%s", best, OUT)
print("Serve it: set WEIGHTS_PATH to", OUT)

## Serve the checkpoint

Copy `skinsense_efficientnet_b0.pt` to `backend/weights/` and set in the backend env:

```
WEIGHTS_PATH=./weights/skinsense_efficientnet_b0.pt
```

It loads into `model_loader.load_model()` with `strict=True` (full TemperatureScaler
state dict). Grad-CAM becomes meaningful immediately.

**Caveats:** `melasma` has no training images and `rosacea` only a handful — the
model outputs 12 logits but will effectively never predict those two. Keep the
UI confidence threshold + disclaimer. Not a medical device.


In [ ]:
# ---- (local helper) zip data/ for Colab upload ----
# Run locally, then upload the resulting data.zip to Colab.
import shutil
shutil.make_archive("data", "zip", ".", "data")
print("wrote data.zip")